## Focus: Alternative ML framing + stronger evaluation discipline

In [1]:
import pandas as pd

df_final = pd.read_csv('../data/targetscan_mirbase_integration_325-3p.csv')

df_final.head()

,miRNA,mean_context_score,min_context_score,n_targets,sequence,length,GC_content,seed,seed_GC_content,seed_group_size
0,hsa-let-7a-5p,-0.282952,-0.809,1386,UGAGGUAGUAGGUUGUAUAGUU,22.0,0.363636,GAGGUAG,0.571429,9
1,hsa-let-7b-5p,-0.281561,-0.809,1386,UGAGGUAGUAGGUUGUGUGGUU,22.0,0.454545,GAGGUAG,0.571429,9
2,hsa-let-7c-5p,-0.283001,-0.809,1386,UGAGGUAGUAGGUUGUAUGGUU,22.0,0.409091,GAGGUAG,0.571429,9
3,hsa-let-7d-5p,-0.294442,-0.827,1386,AGAGGUAGUAGGUUGCAUAGUU,22.0,0.409091,GAGGUAG,0.571429,9
4,hsa-let-7e-5p,-0.282975,-0.809,1386,UGAGGUAGGAGGUUGUAUAGUU,22.0,0.409091,GAGGUAG,0.571429,9


In [2]:
# Reframing our model to address predicted variable (label) as a matter of classificatin rather than regression

# defining label as binary outcome (high vs. low repression)

# strong repressor defined as context score lower than median (more negative = stronger repression)

threshold = df_final['mean_context_score'].median() 
threshold

-0.1870395617772367

In [3]:
# create binary split

df_final['strong_repressor'] = (
    df_final['mean_context_score'] < threshold
).astype(bool)

df_final['strong_repressor'].value_counts(normalize=True)

strong_repressor
False    0.501684
True     0.498316
Name: proportion, dtype: float64

In [4]:
# quickly checking boolean value accross seed group size

df_final.sort_values('seed_group_size', ascending=False).head(15)

,miRNA,mean_context_score,min_context_score,n_targets,sequence,length,GC_content,seed,seed_GC_content,seed_group_size,strong_repressor
193,hsa-miR-372-3p,-0.186694,-0.799,1138,AAAGUGCUGCGACAUUUGAGCGU,23.0,0.478261,AAGUGCU,0.428571,10,False
257,hsa-miR-520d-3p,-0.180916,-0.801,1137,AAAGUGCUUCUCUUUGGUGGGU,22.0,0.454545,AAGUGCU,0.428571,10,False
149,hsa-miR-302a-3p,-0.183029,-0.773,1138,UAAGUGCUUCCAUGUUUUGGUGA,23.0,0.391304,AAGUGCU,0.428571,10,False
150,hsa-miR-302b-3p,-0.183827,-0.781,1138,UAAGUGCUUCCAUGUUUUAGUAG,23.0,0.347826,AAGUGCU,0.428571,10,False
151,hsa-miR-302c-3p,-0.204589,-0.853,1903,UAAGUGCUUCCAUGUUUCAGUGG,23.0,0.434783,AAGUGCU,0.428571,10,True
152,hsa-miR-302d-3p,-0.183065,-0.765,1138,UAAGUGCUUCCAUGUUUGAGUGU,23.0,0.391304,AAGUGCU,0.428571,10,False
153,hsa-miR-302e,-0.180214,-0.759,1137,UAAGUGCUUCCAUGCUU,17.0,0.411765,AAGUGCU,0.428571,10,False
194,hsa-miR-373-3p,-0.130026,-0.729,1137,GAAGUGCUUCGAUUUUGGGGUGU,23.0,0.478261,AAGUGCU,0.428571,10,False
255,hsa-miR-520a-3p,-0.183047,-0.793,1138,AAAGUGCUUCCCUUUGGACUGU,22.0,0.454545,AAGUGCU,0.428571,10,False
256,hsa-miR-520c-3p,-0.184988,-0.815,1138,AAAGUGCUUCCUUUUAGAGGGU,22.0,0.409091,AAGUGCU,0.428571,10,False


In [5]:
# Build logistic regression baseline

import numpy as np
import sklearn

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, RepeatedKFold
from sklearn.pipeline import Pipeline

# define features + label

X = df_final[["length", "GC_content", "seed_GC_content", "seed_group_size"]]
y_class = df_final["strong_repressor"]

# define parameters for GridSearchCV

rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42) #define repeated k-fold

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=5000)) 
])

param_grid = {
    'logreg__C': np.logspace(-3, 3, 50)
    }

grid_log = GridSearchCV(
    pipeline,
    param_grid,
    cv=rkf,
    scoring='roc_auc'
)

grid_log.fit(X, y_class)

,estimator,Pipeline(step..._iter=5000))])
,param_grid,{'logreg__C': array([1.0000...00000000e+03])}
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,RepeatedKFold...ndom_state=42)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [6]:
# scores

print(grid_log.best_params_)
print(grid_log.best_score_)
print(grid_log.cv_results_['std_test_score'][grid_log.best_index_])

{'logreg__C': np.float64(44.98432668969444)}
0.7989081292488858
0.048410169245359326
